# 02 — MGTAB Data Quality

Notebook này kiểm tra chất lượng dữ liệu trước khi xây dựng graph/model. Các kiểm tra chỉ đọc dữ liệu và không thay đổi file raw.

In [13]:
# Nhập thư viện để nạp tensor, lập báo cáo và tính toán số học.
from pathlib import Path

import numpy as np
import pandas as pd
import torch

In [14]:
# Xác định đường dẫn dataset và kiểm tra các raw file bắt buộc.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "MGTAB"
REQUIRED_FILES = [
    "edge_index.pt",# ai đã kết nối với ai
    "edge_type.pt", # loại kết nối
    "edge_weight.pt",# trọng số kết nối
    "features.pt",# đặc trưng của các node
    "labels_bot.pt",# nhãn bot hay không
    "labels_stance.pt",# nhãn stance
]

assert DATA_DIR.is_dir(), f"Không tìm thấy dataset: {DATA_DIR}"
missing_files = [name for name in REQUIRED_FILES if not (DATA_DIR / name).is_file()]
assert not missing_files, f"Thiếu file: {missing_files}"
print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_DIR)

Project root: c:\no-fraud-friend-recommendation
Dataset path: c:\no-fraud-friend-recommendation\data\raw\MGTAB


In [15]:
# Nạp tensor và tạo dictionary chung cho các quality check.
def load_tensor(name):
    path = DATA_DIR / name
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        return torch.load(path, map_location="cpu")# chưa train nên chỉ dùng cpu

edge_index = load_tensor("edge_index.pt")
edge_type = load_tensor("edge_type.pt")
edge_weight = load_tensor("edge_weight.pt")
features = load_tensor("features.pt")
labels_bot = load_tensor("labels_bot.pt")
labels_stance = load_tensor("labels_stance.pt")

objects = {
    "edge_index": edge_index,
    "edge_type": edge_type,
    "edge_weight": edge_weight,
    "features": features,
    "labels_bot": labels_bot,
    "labels_stance": labels_stance,
}

pd.DataFrame({
    "object": list(objects),
    "shape": [tuple(value.shape) for value in objects.values()],
    "dtype": [str(value.dtype) for value in objects.values()],
})

,object,shape,dtype
0,edge_index,"(2, 1700108)",torch.int64
1,edge_type,"(1700108,)",torch.int64
2,edge_weight,"(1700108,)",torch.float32
3,features,"(10199, 788)",torch.float32
4,labels_bot,"(10199,)",torch.int64
5,labels_stance,"(10199,)",torch.int64


- feature(10199, 788) : 10,199 node và mỗi node sẽ được biểu diên bằng 1 vector 788 chiều ( 788 features/node) và có thể biểu diễn features[user_id]
- edge_index :có shape (2, 1,700,108) → tức là có 1,700,108 cạnh trong đồ thị(Sẽ là 1 ma trận 2x1700108) trong đó hàng 1 = source nodes và hàng 2 = target nodes
=> mỗi cột trong edge_index là 1 cạnh (source => target)


## 1. Kiểm tra cấu trúc và tính nhất quán

In [ ]:
# Kiểm tra shape, độ dài và range của node, edge, relation và label.

num_nodes = features.shape[0]
num_edges = edge_index.shape[1]
src, dst = edge_index[0], edge_index[1]

checks = {
    "edge_index_shape": edge_index.ndim == 2 and edge_index.shape[0] == 2,
    "edge_type_length": edge_type.shape == (num_edges,),
    "edge_weight_length": edge_weight.shape == (num_edges,),
    "edge_type_values_valid": bool(torch.isin(edge_type, torch.arange(7)).all()),
    "bot_label_values_valid": bool(torch.isin(labels_bot, torch.tensor([0, 1])).all()),
    "stance_label_values_valid": bool(torch.isin(labels_stance, torch.tensor([0, 1, 2])).all()),
    "bot_label_length": labels_bot.shape == (num_nodes,),
    "stance_label_length": labels_stance.shape == (num_nodes,),
    "edge_ids_non_negative": int(edge_index.min()) >= 0,
    "edge_ids_in_range": int(edge_index.max()) < num_nodes,
}

quality_report = pd.DataFrame([
    {"check": name, "passed": passed}
    for name, passed in checks.items()
])
quality_report

,check,passed
0,edge_index_shape,True
1,edge_type_length,True
2,edge_weight_length,True
3,edge_type_values_valid,True
4,bot_label_values_valid,True
5,stance_label_values_valid,True
6,bot_label_length,True
7,stance_label_length,True
8,edge_ids_non_negative,True
9,edge_ids_in_range,True


In [17]:
# Tạo hàm kiểm tra NaN, Inf và miền giá trị của tensor số.
def numeric_quality(name, tensor):
    return {
        "object": name,
        "has_nan": bool(torch.isnan(tensor).any()),
        "has_inf": bool(torch.isinf(tensor).any()),
        "min": float(tensor.min()),
        "max": float(tensor.max()),
    }

numeric_report = pd.DataFrame([
    numeric_quality("features", features),
    numeric_quality("edge_weight", edge_weight),
])
numeric_report

,object,has_nan,has_inf,min,max
0,features,False,False,-6.205815,10.199645
1,edge_weight,False,False,0.025262,1.000000


## 2. Phân phối quan hệ, trọng số và nhãn

In [18]:
# Tạo bảng phân phối relation và label để tìm mất cân bằng hoặc giá trị lạ.
RELATION_NAMES = {
    0: "followers", 1: "friends", 2: "mention",
    3: "reply", 4: "quoted", 5: "url", 6: "hashtag",
}

def distribution(tensor, value_name):
    values, counts = torch.unique(tensor, return_counts=True)
    return pd.DataFrame({
        value_name: values.tolist(),
        "count": counts.tolist(),
        "share": (counts.float() / tensor.numel()).tolist(),
    })

relation_report = distribution(edge_type, "relation_id")
relation_report["relation"] = relation_report["relation_id"].map(RELATION_NAMES).fillna("unknown")
bot_report = distribution(labels_bot, "bot_label")
stance_report = distribution(labels_stance, "stance_label")

print("Relations")
display(relation_report)
print("Bot labels")
display(bot_report)
print("Stance labels")
display(stance_report)

Relations


,relation_id,count,share,relation
0,0,308120,0.181236,followers
1,1,412575,0.242676,friends
2,2,114516,0.067358,mention
3,3,223466,0.131442,reply
4,4,77631,0.045662,quoted
5,5,263800,0.155167,url
6,6,300000,0.176459,hashtag


Bot labels


,bot_label,count,share
0,0,7451,0.730562
1,1,2748,0.269438


Stance labels


,stance_label,count,share
0,0,3776,0.370232
1,1,3637,0.356604
2,2,2786,0.273164


In [19]:
# Kiểm tra duplicate edge, self-loop, node degree và edge weight bất thường.
unique_edges = torch.unique(edge_index.t(), dim=0).shape[0]
out_degree = torch.bincount(src, minlength=num_nodes)
in_degree = torch.bincount(dst, minlength=num_nodes)
total_degree = out_degree + in_degree

graph_report = pd.DataFrame({
    "metric": [
        "num_nodes", "num_edges", "unique_edges", "duplicate_edges",
        "self_loops", "isolated_nodes", "zero_weight_edges", "negative_weight_edges",
    ],
    "value": [
        num_nodes, num_edges, unique_edges, num_edges - unique_edges,
        int((src == dst).sum()), int((total_degree == 0).sum()),
        int((edge_weight == 0).sum()), int((edge_weight < 0).sum()),
    ],
})

degree_report = pd.DataFrame({
    "degree_type": ["in", "out", "total"],
    "min": [int(in_degree.min()), int(out_degree.min()), int(total_degree.min())],
    "median": [float(in_degree.median()), float(out_degree.median()), float(total_degree.median())],
    "max": [int(in_degree.max()), int(out_degree.max()), int(total_degree.max())],
})
display(graph_report)
display(degree_report)

,metric,value
0,num_nodes,10199
1,num_edges,1700108
2,unique_edges,1138182
3,duplicate_edges,561926
4,self_loops,205819
5,isolated_nodes,52
6,zero_weight_edges,0
7,negative_weight_edges,0


,degree_type,min,median,max
0,in,0,66.0,8330
1,out,0,83.0,8383
2,total,0,153.0,16713


In [20]:
# Đếm feature hằng và feature gần hằng để đánh giá chất lượng feature.
feature_std = features.float().std(dim=0)
constant_features = int((feature_std == 0).sum())
near_constant_features = int((feature_std < 1e-8).sum())

feature_report = pd.DataFrame({
    "metric": ["num_features", "constant_features", "near_constant_features"],
    "value": [features.shape[1], constant_features, near_constant_features],
})
feature_report

,metric,value
0,num_features,788
1,constant_features,0
2,near_constant_features,0


## 3. Kết luận

Dựa trên các bảng ở trên, cần đặc biệt xem xét duplicate edges, self-loops, isolated nodes, nhãn mất cân bằng, edge weight bất thường và feature hằng. Chỉ thực hiện cleaning sau khi xác định ý nghĩa của từng loại dữ liệu với bài toán.